In [1]:
import pandas as pd
import sqlite3 

In [3]:
df = pd.read_csv('Superstore.csv', encoding='latin-1')
df['Order Date'] = pd.to_datetime(df['Order Date'])

conn = sqlite3.connect('superstore.db')
df.to_sql('superstore', conn, if_exists='replace', index=False)
conn.close()

print("Database created")

Database created


In [9]:
conn = sqlite3.connect('superstore.db')
query = '''
SELECT 
    Category,
    SUM(Sales) AS total_sales,
    SUM(Profit) AS total_profit,
    SUM(Quantity) AS total_quantity,
    SUM(Sales) - SUM(Profit) AS total_cost
FROM 
    superstore 
GROUP BY 
    Category
ORDER BY 
    total_sales DESC;
'''

result = pd.read_sql_query(query, conn)
conn.close()

result

,Category,total_sales,total_profit,total_quantity,total_cost
0,Technology,836154.0330,145454.9481,6939,690699.0849
1,Furniture,741999.7953,18451.2728,8028,723548.5225
2,Office Supplies,719047.0320,122490.8008,22906,596556.2312


In [14]:
conn = sqlite3.connect('superstore.db')

query = '''
    WITH Customer_top AS (
        SELECT 
            [Customer Name]            AS Customer_name,
            SUM(Sales)                 AS Total_sales,
            SUM(Quantity)              AS Total_quantity,
            COUNT(DISTINCT [Order ID]) AS Total_orders
        FROM superstore
        GROUP BY [Customer ID]
    )
    SELECT 
        Customer_name,
        Total_sales,
        Total_quantity,
        Total_orders,
        RANK() OVER (ORDER BY Total_sales DESC) AS Sales_rank
    FROM Customer_top
    LIMIT 10;
'''

results = pd.read_sql_query(query, conn)
conn.close()  

print(results)

        Customer_name  Total_sales  Total_quantity  Total_orders  Sales_rank
0         Sean Miller    25043.050              50             5           1
1        Tamara Chand    19052.218              42             5           2
2        Raymond Buch    15117.339              71             6           3
3        Tom Ashbrook    14595.620              36             4           4
4       Adrian Barton    14473.571              73            10           5
5        Ken Lonsdale    14175.229             113            12           6
6        Sanjit Chand    14142.334              87             9           7
7        Hunter Lopez    12873.298              50             6           8
8        Sanjit Engle    12209.438              78            11           9
9  Christopher Conant    12129.072              34             5          10


In [21]:
conn = sqlite3.connect('superstore.db')

query = '''
    WITH Monthly_Sales AS (
        SELECT 
            strftime('%Y-%m', [Order Date]) AS Year_Month,
            SUM(Sales)    AS total_sales,
            AVG(Profit)   AS avg_profit,
            SUM(Quantity) AS total_quantity
        FROM superstore
        GROUP BY Year_Month
    ),
    Ranked AS (
        SELECT 
            Year_Month,
            total_sales,
            avg_profit,
            total_quantity,
            RANK() OVER (ORDER BY total_sales DESC) AS sales_rank
        FROM Monthly_Sales
    )
    SELECT * FROM Ranked
    WHERE sales_rank <= 5;
'''

results = pd.read_sql_query(query, conn)
conn.close()

print(results)

  Year_Month  total_sales  avg_profit  total_quantity  sales_rank
0    2017-11  118447.8250   21.111337            1840           1
1    2016-12   96999.0430   50.810538            1414           2
2    2017-09   87866.6520   23.946744            1660           3
3    2017-12   83829.3188   18.362223            1723           4
4    2014-09   81777.3508   31.074998            1000           5


In [30]:
conn = sqlite3.connect('superstore.db')

query = '''
    SELECT 
        [Segment],
        [Category],
        SUM(Sales)  AS total_sales,
        SUM(Profit) AS total_profit
    FROM superstore
    GROUP BY [Segment], [Category]
'''

results = pd.read_sql_query(query, conn)
conn.close()

pivot = results.pivot_table(
    index='Segment',
    columns='Category',
    values=['total_sales', 'total_profit'],
    aggfunc='sum'
).round(2)

print(pivot)

            total_profit                            total_sales  \
Category       Furniture Office Supplies Technology   Furniture   
Segment                                                           
Consumer         6991.08        56330.32   70797.81   391049.31   
Corporate        7584.82        40227.32   44167.00   229019.79   
Home Office      3875.38        25933.16   30490.14   121930.70   

                                        
Category    Office Supplies Technology  
Segment                                 
Consumer          363952.14  406399.90  
Corporate         230676.46  246450.12  
Home Office       124418.43  183304.02  


In [38]:
conn = sqlite3.connect('superstore.db') 
query = ''' 
    SELECT 
        Region ,  
        SUM(Sales) AS Total_Sales , 
        SUM(Quantity) AS Total_Quantity , 
        Sum(Profit) AS Total_Profit 
    FROM 
        superstore 
    GROUP BY Region
    ORDER BY Total_Sales DESC;
'''

results = pd.read_sql_query(query,conn) 
conn.close() 
print(results)

    Region  Total_Sales  Total_Quantity  Total_Profit
0     West  725457.8245           12266   108418.4489
1     East  678781.2400           10618    91522.7800
2  Central  501239.8908            8780    39706.3625
3    South  391721.9050            6209    46749.4303
